# Comparativo de modelos preentrenados para detectar daño en calles
**Proyecto UrbanOS · Visión por computadora · Maestría en IA**

**Caso de uso.** UrbanOS es una plataforma municipal para revisar el estado de las calles. Antes de contar y ubicar cada bache, el sistema debe decidir si una foto muestra un bache o una calle sin daño. En este cuaderno comparo tres modelos preentrenados de Hugging Face en esa tarea.

**Tarea.** Clasificación binaria de imágenes: `bache` (clase positiva) y `sin_dano`.

**Ajuste mínimo.** Los modelos no se reentrenan. Los uso congelados como extractores de características y entreno encima una regresión logística.

**Entorno.** Google Colab con GPU T4 y un token de lectura de Hugging Face guardado en los Secretos de Colab (`HF_TOKEN`).

## 1. Configuración de GPU

In [ ]:
import platform, sys, torch

print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
HAY_GPU = torch.cuda.is_available()
DISPOSITIVO = torch.device("cuda" if HAY_GPU else "cpu")
print("GPU:", torch.cuda.get_device_name(0) if HAY_GPU else "no disponible, se usa CPU")

## 2. Instalación de dependencias

In [ ]:
!pip install -q transformers datasets huggingface_hub scikit-learn pandas matplotlib openpyxl pillow

## 3. Parámetros del experimento

In [ ]:
import os, json, random, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

SEMILLA = 42
MAX_POR_CLASE = None
MEDIR_CPU = True
N_LATENCIA = 100
ANCHO_LOTE = 32

MODELOS = {
    "MobileNetV2": "google/mobilenet_v2_1.0_224",
    "ResNet-50":   "microsoft/resnet-50",
    "ViT-Base":    "google/vit-base-patch16-224",
}

FUENTE_DATASET = {
    "nombre": "Pothole Detection Dataset",
    "autor": "Atulya Kumar",
    "url": "https://www.kaggle.com/datasets/atulyakumar98/pothole-detection-dataset",
    "licencia": "CC0: Public Domain",
    "fecha_descarga": "2026-10-10",
    "descripcion": "681 archivos en dos carpetas: 'normal' (352 imágenes de calles lisas) y 'potholes' (calles con baches). Según el autor, las imágenes se tomaron de internet para un proyecto universitario.",
}

RUTA_ZIP = "/content/dataset.zip"

PROYECTO = Path("/content/proyecto")
RAW = PROYECTO / "data" / "raw"
RESULTADOS = PROYECTO / "results"
for carpeta in (RAW, RESULTADOS, PROYECTO / "src"):
    carpeta.mkdir(parents=True, exist_ok=True)
os.chdir(PROYECTO)

random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)

## 4. Autenticación en Hugging Face Hub
Inicio sesión con un token de lectura guardado en los Secretos de Colab (`HF_TOKEN`).

In [ ]:
from huggingface_hub import login, whoami

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception as error:
    print("No se encontró el secreto HF_TOKEN:", type(error).__name__)
    from huggingface_hub import notebook_login
    notebook_login()

print("Sesión iniciada en Hugging Face como:", whoami()["name"])

## 5. Dataset
Uso el *Pothole Detection Dataset* de Atulya Kumar, publicado en Kaggle con licencia CC0: 681 archivos en dos carpetas, `normal` y `potholes`. Descargué el ZIP y lo subí a Colab como `dataset.zip`.

La clase de cada imagen se deduce del nombre de su carpeta. Quito imágenes dañadas, duplicados exactos y duplicados con etiquetas contradictorias, y hago una división propia `train / val / test` de 70 / 15 / 15, con semilla fija y proporción de clases conservada. El código de preparación está en `src/datos.py` y `src/metricas.py`.

In [ ]:
if not any(RAW.rglob("*")):
    if not Path(RUTA_ZIP).exists():
        raise FileNotFoundError(f"No se encontró {RUTA_ZIP}.")
    with zipfile.ZipFile(RUTA_ZIP) as z:
        z.extractall(RAW)

print("Archivos en data/raw:", sum(1 for ruta in RAW.rglob("*") if ruta.is_file()))

In [ ]:
%%writefile src/datos.py
"""Preparación del dataset de baches."""
import hashlib
from pathlib import Path

import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASES = ["sin_dano", "bache"]
PALABRAS_SIN_DANO = ("normal", "plain", "good", "no_pothole", "nopothole", "sin_dano", "sano")
PALABRAS_BACHE = ("pothole", "bache")

DICCIONARIO = [
    ("id", "texto", "Identificador de la imagen dentro del dataset (img_00001, img_00002, ...)."),
    ("ruta_relativa", "texto", "Ruta de la imagen dentro de la carpeta data/raw."),
    ("clase", "texto", "Clase de la imagen: 'bache' o 'sin_dano', deducida del nombre de la carpeta original."),
    ("etiqueta", "entero", "Clase en número: 1 = bache, 0 = sin_dano."),
    ("particion", "texto", "Parte del dataset: 'train', 'val' o 'test'."),
    ("ancho_px", "entero", "Ancho de la imagen en píxeles."),
    ("alto_px", "entero", "Alto de la imagen en píxeles."),
    ("tamano_kb", "decimal", "Tamaño del archivo en kilobytes."),
    ("md5", "texto", "Huella del archivo, usada para detectar imágenes repetidas."),
]
COLUMNAS = [campo for campo, _, _ in DICCIONARIO]


def clase_desde_ruta(ruta_relativa):
    """Deduce la clase a partir de los nombres de las carpetas."""
    partes = [p.lower() for p in Path(ruta_relativa).parts[:-1]]
    for parte in partes:
        if any(palabra in parte for palabra in PALABRAS_SIN_DANO):
            return "sin_dano"
    for parte in partes:
        if any(palabra in parte for palabra in PALABRAS_BACHE):
            return "bache"
    return None


def escanear_imagenes(raiz):
    """Lista las imágenes de la carpeta con su clase."""
    raiz = Path(raiz)
    filas = []
    for ruta in sorted(raiz.rglob("*")):
        if not ruta.is_file() or ruta.suffix.lower() not in EXTENSIONES:
            continue
        relativa = ruta.relative_to(raiz)
        clase = clase_desde_ruta(relativa)
        if clase is not None:
            filas.append({"ruta_relativa": relativa.as_posix(), "clase": clase})
    return pd.DataFrame(filas, columns=["ruta_relativa", "clase"])


def agregar_metadatos(df, raiz):
    """Agrega tamaño, dimensiones y huella md5 de cada imagen."""
    raiz = Path(raiz)
    filas = []
    for relativa in df["ruta_relativa"]:
        ruta = raiz / relativa
        contenido = ruta.read_bytes()
        ancho = alto = None
        valida = True
        try:
            with Image.open(ruta) as imagen:
                imagen.load()
                ancho, alto = imagen.size
        except Exception:
            valida = False
        filas.append({
            "md5": hashlib.md5(contenido).hexdigest(),
            "tamano_kb": round(len(contenido) / 1024, 1),
            "ancho_px": ancho,
            "alto_px": alto,
            "valida": valida,
        })
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(filas)], axis=1)


def depurar(df):
    """Quita imágenes dañadas, duplicados exactos y duplicados con clases contradictorias."""
    resumen = {"imagenes_encontradas": int(len(df))}

    df = df[df["valida"]].copy()
    resumen["invalidas_eliminadas"] = resumen["imagenes_encontradas"] - int(len(df))

    clases_por_huella = df.groupby("md5")["clase"].nunique()
    contradictorias = clases_por_huella[clases_por_huella > 1].index
    antes = len(df)
    df = df[~df["md5"].isin(contradictorias)].copy()
    resumen["contradictorias_eliminadas"] = int(antes - len(df))

    antes = len(df)
    df = df.drop_duplicates(subset="md5", keep="first").copy()
    resumen["duplicadas_eliminadas"] = int(antes - len(df))

    df = df.drop(columns="valida").reset_index(drop=True)
    df["ancho_px"] = df["ancho_px"].astype(int)
    df["alto_px"] = df["alto_px"].astype(int)
    resumen["imagenes_finales"] = int(len(df))
    return df, resumen


def dividir(df, semilla=42, prop_val=0.15, prop_test=0.15):
    """Divide en train, val y test conservando la proporción de clases."""
    df = df.copy()
    resto, test = train_test_split(df, test_size=prop_test, stratify=df["clase"], random_state=semilla)
    train, val = train_test_split(
        resto, test_size=prop_val / (1 - prop_test), stratify=resto["clase"], random_state=semilla
    )
    df.loc[train.index, "particion"] = "train"
    df.loc[val.index, "particion"] = "val"
    df.loc[test.index, "particion"] = "test"
    return df


def construir_manifiesto(raiz, semilla=42, max_por_clase=None):
    """Escanea, depura y divide el dataset; devuelve el manifiesto y un resumen."""
    df = escanear_imagenes(raiz)
    if df.empty:
        raise ValueError(f"No se encontraron imágenes clasificables en {raiz}.")
    conteo = df["clase"].value_counts()
    if len(conteo) < 2:
        raise ValueError(f"Solo se encontró una clase ({conteo.index[0]}).")

    if max_por_clase:
        df = pd.concat(
            [g.sample(min(len(g), max_por_clase), random_state=semilla) for _, g in df.groupby("clase")]
        )

    df = agregar_metadatos(df, raiz)
    df, resumen = depurar(df)
    if df["clase"].value_counts().min() < 10:
        raise ValueError("Hay menos de 10 imágenes en alguna clase.")

    df = dividir(df, semilla=semilla).sort_values("ruta_relativa").reset_index(drop=True)
    df["id"] = [f"img_{i + 1:05d}" for i in range(len(df))]
    df["etiqueta"] = df["clase"].map({"sin_dano": 0, "bache": 1}).astype(int)
    resumen["semilla"] = semilla
    resumen["max_por_clase"] = max_por_clase
    return df[COLUMNAS], resumen

In [ ]:
%%writefile src/metricas.py
"""Métricas de clasificación binaria (clase positiva: bache)."""
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score


def calcular_metricas(y_real, prob_bache, umbral=0.5):
    """Accuracy, precision, recall, F1 y matriz de confusión para un umbral."""
    y_real = np.asarray(y_real)
    y_pred = (np.asarray(prob_bache) >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(y_real, y_pred, zero_division=0),
        "recall": recall_score(y_real, y_pred, zero_division=0),
        "f1": f1_score(y_real, y_pred, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def tabla_umbrales(y_real, prob_bache, umbrales=(0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8)):
    """Métricas para varios umbrales de decisión."""
    filas = []
    for umbral in umbrales:
        fila = calcular_metricas(y_real, prob_bache, umbral)
        fila["umbral"] = umbral
        filas.append(fila)
    return pd.DataFrame(filas)[["umbral", "accuracy", "precision", "recall", "f1", "fp", "fn"]]

In [ ]:
Path("src/__init__.py").write_text("")

import sys
sys.path.insert(0, str(PROYECTO))
from src.datos import construir_manifiesto, DICCIONARIO, CLASES
from src.metricas import calcular_metricas, tabla_umbrales

manifiesto, resumen = construir_manifiesto(RAW, semilla=SEMILLA, max_por_clase=MAX_POR_CLASE)
print(json.dumps(resumen, indent=2, ensure_ascii=False))
display(manifiesto.head())
display(pd.crosstab(manifiesto["particion"], manifiesto["clase"], margins=True))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, ejes = plt.subplots(1, 5, figsize=(16, 3.2))
manifiesto["clase"].value_counts().reindex(CLASES).plot.bar(ax=ejes[0], color=["#16A34A", "#DC2626"])
ejes[0].set_title("Imágenes por clase"); ejes[0].set_xlabel("")
ejemplos = pd.concat([g.sample(2, random_state=SEMILLA) for _, g in manifiesto.groupby("clase")])
for eje, (_, fila) in zip(ejes[1:], ejemplos.iterrows()):
    eje.imshow(Image.open(RAW / fila["ruta_relativa"]).convert("RGB")); eje.set_title(fila["clase"]); eje.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
(PROYECTO / "data").mkdir(exist_ok=True)
manifiesto.to_csv(PROYECTO / "data" / "dataset.csv", index=False)
diccionario = pd.DataFrame(DICCIONARIO, columns=["campo", "tipo", "descripcion"])
diccionario.to_csv(PROYECTO / "data" / "diccionario_datos.csv", index=False)
(PROYECTO / "data" / "origen_dataset.json").write_text(json.dumps(FUENTE_DATASET, indent=2, ensure_ascii=False), encoding="utf-8")
(PROYECTO / "data" / "resumen_dataset.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False), encoding="utf-8")

with pd.ExcelWriter(PROYECTO / "data" / "dataset_baches.xlsx") as excel:
    manifiesto.to_excel(excel, sheet_name="dataset", index=False)
    diccionario.to_excel(excel, sheet_name="diccionario", index=False)
    pd.DataFrame(list(FUENTE_DATASET.items()), columns=["campo", "valor"]).to_excel(excel, sheet_name="origen", index=False)
    pd.DataFrame(list(resumen.items()), columns=["concepto", "valor"]).to_excel(excel, sheet_name="resumen", index=False)
display(diccionario)

### Carga del dataset con la librería `datasets` de Hugging Face

In [ ]:
from datasets import Dataset, Image as ImagenHF

dfs = {p: manifiesto[manifiesto["particion"] == p].reset_index(drop=True) for p in ("train", "val", "test")}

def crear_dataset_hf(df):
    rutas = [str(RAW / r) for r in df["ruta_relativa"]]
    return Dataset.from_dict({"image": rutas, "label": df["etiqueta"].tolist()}).cast_column("image", ImagenHF())

dsets = {p: crear_dataset_hf(df) for p, df in dfs.items()}
print({p: len(d) for p, d in dsets.items()})

## 6. Modelos candidatos
Elegí tres modelos de tamaños muy distintos para observar el equilibrio entre eficiencia y precisión: MobileNetV2 (ligero), ResNet-50 (mediano) y ViT-Base (grande, basado en transformers). La licencia y la tarea de cada uno se consultan directamente al Hub.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
def info_modelo(model_id):
    info = api.model_info(model_id)
    licencia = getattr(info.card_data, "license", None) if info.card_data is not None else None
    return {"id_hub": model_id, "tarea_hub": info.pipeline_tag, "licencia_hub": licencia}

info_hub = {nombre: info_modelo(model_id) for nombre, model_id in MODELOS.items()}
display(pd.DataFrame(info_hub).T)

## 7. Pipeline de inferencia, ajuste mínimo y medición
Para cada modelo:
1. Cargo el modelo y su procesador de imágenes desde el Hub.
2. Extraigo los vectores de características de `train`, `val` y `test` con el modelo congelado.
3. Entreno una regresión logística sobre esos vectores y elijo su parámetro `C` con la partición `val`.
4. Evalúo en `test` con accuracy, precision, recall, F1 y AUC.
5. Mido la latencia por imagen en GPU y CPU, la velocidad por lote, la memoria de GPU y el tamaño del modelo.

In [ ]:
from transformers import AutoImageProcessor, AutoModel
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, roc_auc_score

def crear_cargador(ds, procesador):
    def agrupar(lote):
        imagenes = [ej["image"].convert("RGB") for ej in lote]
        pixeles = procesador(images=imagenes, return_tensors="pt")["pixel_values"]
        return pixeles, torch.tensor([ej["label"] for ej in lote])
    return DataLoader(ds, batch_size=ANCHO_LOTE, shuffle=False, num_workers=2, collate_fn=agrupar)

@torch.no_grad()
def vector_caracteristicas(modelo, pixeles):
    salida = modelo(pixel_values=pixeles)
    if modelo.config.model_type == "vit":
        return salida.last_hidden_state[:, 0]
    return salida.pooler_output.flatten(1)

@torch.no_grad()
def extraer(modelo, ds, procesador):
    vectores, etiquetas = [], []
    for pixeles, y in crear_cargador(ds, procesador):
        v = vector_caracteristicas(modelo, pixeles.to(DISPOSITIVO))
        vectores.append(v.float().cpu().numpy()); etiquetas.append(y.numpy())
    return np.concatenate(vectores), np.concatenate(etiquetas)

def entrenar_cabeza(X_train, y_train, X_val, y_val):
    mejor = None
    for C in (0.01, 0.1, 1.0, 10.0):
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000, class_weight="balanced"))
        clf.fit(X_train, y_train)
        f1 = f1_score(y_val, clf.predict(X_val), zero_division=0)
        if mejor is None or f1 > mejor[0]:
            mejor = (f1, C, clf)
    return mejor[2], mejor[1], mejor[0]

@torch.no_grad()
def una_inferencia(modelo, procesador, cabeza, imagen, dispositivo):
    pixeles = procesador(images=imagen, return_tensors="pt")["pixel_values"].to(dispositivo)
    v = vector_caracteristicas(modelo, pixeles).float().cpu().numpy()
    return cabeza.predict_proba(v)[0, 1]

def medir_latencia(modelo, procesador, cabeza, imagenes, dispositivo):
    modelo.to(dispositivo)
    for im in imagenes[:5]:
        una_inferencia(modelo, procesador, cabeza, im, dispositivo)
    tiempos = []
    for im in imagenes:
        t0 = time.perf_counter()
        una_inferencia(modelo, procesador, cabeza, im, dispositivo)
        tiempos.append((time.perf_counter() - t0) * 1000)
    return {"mediana_ms": float(np.median(tiempos)), "p95_ms": float(np.percentile(tiempos, 95))}

@torch.no_grad()
def medir_rendimiento(modelo, procesador, imagenes):
    lote = procesador(images=imagenes[:ANCHO_LOTE], return_tensors="pt")["pixel_values"].to(DISPOSITIVO)
    for _ in range(2):
        vector_caracteristicas(modelo, lote)
    if HAY_GPU:
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    repeticiones = 5
    t0 = time.perf_counter()
    for _ in range(repeticiones):
        vector_caracteristicas(modelo, lote)
    if HAY_GPU:
        torch.cuda.synchronize()
    segundos = time.perf_counter() - t0
    return {
        "imgs_por_seg": len(lote) * repeticiones / segundos,
        "vram_pico_mb": torch.cuda.max_memory_allocated() / 1e6 if HAY_GPU else None,
    }

In [ ]:
n_lat = min(N_LATENCIA, len(dsets["test"]))
imagenes_lat = [dsets["test"][i]["image"].convert("RGB") for i in range(n_lat)]

filas, probs = [], {}
for nombre, model_id in MODELOS.items():
    print(f"\n=== {nombre} ({model_id}) ===")
    procesador = AutoImageProcessor.from_pretrained(model_id)
    modelo = AutoModel.from_pretrained(model_id).to(DISPOSITIVO).eval()
    n_param = sum(p.numel() for p in modelo.parameters())

    t0 = time.perf_counter()
    X, y = {}, {}
    for parte in ("train", "val", "test"):
        X[parte], y[parte] = extraer(modelo, dsets[parte], procesador)
    t_extraccion = time.perf_counter() - t0

    cabeza, mejor_C, f1_val = entrenar_cabeza(X["train"], y["train"], X["val"], y["val"])
    prob_test = cabeza.predict_proba(X["test"])[:, 1]
    probs[nombre] = prob_test

    m = calcular_metricas(y["test"], prob_test)
    lat_gpu = medir_latencia(modelo, procesador, cabeza, imagenes_lat, DISPOSITIVO)
    rend = medir_rendimiento(modelo, procesador, imagenes_lat)
    lat_cpu = None
    if MEDIR_CPU:
        modelo.to("cpu")
        lat_cpu = medir_latencia(modelo, procesador, cabeza, imagenes_lat[:30], torch.device("cpu"))

    filas.append({
        "modelo": nombre, "id_hub": model_id, "licencia": info_hub[nombre]["licencia_hub"],
        "parametros_M": n_param / 1e6, "tamano_MB_fp32": n_param * 4 / 1e6,
        "accuracy": m["accuracy"], "precision": m["precision"], "recall": m["recall"], "f1": m["f1"],
        "auc": roc_auc_score(y["test"], prob_test), "C_elegido": mejor_C, "f1_validacion": f1_val,
        "lat_gpu_mediana_ms": lat_gpu["mediana_ms"], "lat_gpu_p95_ms": lat_gpu["p95_ms"],
        "lat_cpu_mediana_ms": lat_cpu["mediana_ms"] if lat_cpu else None,
        "imgs_por_seg_gpu": rend["imgs_por_seg"], "vram_pico_mb": rend["vram_pico_mb"],
        "tiempo_extraccion_s": t_extraccion, "tn": m["tn"], "fp": m["fp"], "fn": m["fn"], "tp": m["tp"],
    })
    print(f"F1={m['f1']:.4f} | accuracy={m['accuracy']:.4f} | latencia GPU={lat_gpu['mediana_ms']:.1f} ms")
    del modelo
    if HAY_GPU:
        torch.cuda.empty_cache()

## 8. Resultados

In [ ]:
tabla = pd.DataFrame(filas)
tabla.to_csv(RESULTADOS / "metricas_modelos.csv", index=False)
resumen_tabla = tabla[["modelo", "licencia", "parametros_M", "tamano_MB_fp32", "accuracy", "precision", "recall", "f1",
                       "auc", "lat_gpu_mediana_ms", "lat_cpu_mediana_ms", "imgs_por_seg_gpu", "vram_pico_mb"]]
display(resumen_tabla.round(3))

In [ ]:
import matplotlib.pyplot as plt

ax = tabla.set_index("modelo")[["accuracy", "precision", "recall", "f1"]].plot.bar(figsize=(8, 4), rot=0, ylim=(0, 1.05))
ax.set_title("Métricas en el conjunto de prueba"); ax.legend(loc="lower right"); plt.tight_layout()
plt.savefig(RESULTADOS / "metricas_por_modelo.png", dpi=150); plt.show()

fig, eje = plt.subplots(figsize=(6, 4))
eje.scatter(tabla["lat_gpu_mediana_ms"], tabla["f1"], s=tabla["parametros_M"].clip(lower=5) * 4 + 40)
for _, f in tabla.iterrows():
    eje.annotate(f["modelo"], (f["lat_gpu_mediana_ms"], f["f1"]), textcoords="offset points", xytext=(6, 6))
eje.set_xlabel("Latencia mediana en GPU (ms por imagen)"); eje.set_ylabel("F1"); eje.set_title("Eficiencia frente a precisión")
plt.tight_layout(); plt.savefig(RESULTADOS / "eficiencia_vs_precision.png", dpi=150); plt.show()

fig, ejes = plt.subplots(1, len(tabla), figsize=(4 * len(tabla), 3.5))
ejes = np.atleast_1d(ejes)
for eje, (_, f) in zip(ejes, tabla.iterrows()):
    matriz = np.array([[f["tn"], f["fp"]], [f["fn"], f["tp"]]])
    eje.imshow(matriz, cmap="Blues")
    for i in range(2):
        for j in range(2):
            eje.text(j, i, int(matriz[i, j]), ha="center", va="center")
    eje.set_xticks([0, 1], ["sin_dano", "bache"]); eje.set_yticks([0, 1], ["sin_dano", "bache"])
    eje.set_xlabel("Predicho"); eje.set_ylabel("Real"); eje.set_title(f["modelo"])
plt.tight_layout(); plt.savefig(RESULTADOS / "matrices_confusion.png", dpi=150); plt.show()

## 9. Umbrales de decisión
En UrbanOS no detectar un bache real pesa más que una falsa alarma, por eso reviso cómo cambian precision y recall al mover el umbral con el que se decide que hay un bache.

In [ ]:
y_test = dfs["test"]["etiqueta"].to_numpy()
umbrales = pd.concat(
    [tabla_umbrales(y_test, p).assign(modelo=nombre) for nombre, p in probs.items()], ignore_index=True
)
umbrales.to_csv(RESULTADOS / "umbrales_modelos.csv", index=False)

mejor = tabla.loc[tabla["f1"].idxmax(), "modelo"]
print("Modelo con mejor F1:", mejor)
display(umbrales[umbrales["modelo"] == mejor].round(3))

## 10. Errores del mejor modelo

In [ ]:
prob_mejor = probs[mejor]
prueba = dfs["test"].assign(prob=prob_mejor)
falsos_negativos = prueba[(prueba["etiqueta"] == 1) & (prueba["prob"] < 0.5)].sort_values("prob")
falsos_positivos = prueba[(prueba["etiqueta"] == 0) & (prueba["prob"] >= 0.5)].sort_values("prob", ascending=False)
print(f"Errores de {mejor} en prueba ({len(prueba)} imágenes): {len(falsos_negativos)} falsos negativos, "
      f"{len(falsos_positivos)} falsas alarmas")

fig, ejes = plt.subplots(2, 4, figsize=(14, 6))
for fila_ejes, (titulo, datos) in zip(ejes, [("Bache NO detectado", falsos_negativos), ("Falsa alarma", falsos_positivos)]):
    for eje in fila_ejes:
        eje.axis("off")
    if datos.empty:
        fila_ejes[0].text(0, 0.5, f"{titulo}: ninguno en la prueba", fontsize=11, transform=fila_ejes[0].transAxes)
    for eje, (_, f) in zip(fila_ejes, datos.head(4).iterrows()):
        eje.imshow(Image.open(RAW / f["ruta_relativa"]).convert("RGB"))
        eje.set_title(f"{titulo}\nprob. bache = {f['prob']:.2f}", fontsize=9)
plt.tight_layout(); plt.savefig(RESULTADOS / "errores_mejor_modelo.png", dpi=150); plt.show()

## 11. Conclusiones parciales

In [ ]:
mejor_f1 = tabla.loc[tabla["f1"].idxmax()]
mas_rapido = tabla.loc[tabla["lat_gpu_mediana_ms"].idxmin()]
mas_ligero = tabla.loc[tabla["tamano_MB_fp32"].idxmin()]
mayor_recall = tabla.loc[tabla["recall"].idxmax()]

print(f"• Mejor F1: {mejor_f1['modelo']} ({mejor_f1['f1']:.3f}), con accuracy {mejor_f1['accuracy']:.3f}.")
print(f"• Mayor recall: {mayor_recall['modelo']} ({mayor_recall['recall']:.3f}).")
print(f"• Más rápido en GPU: {mas_rapido['modelo']} ({mas_rapido['lat_gpu_mediana_ms']:.1f} ms por imagen).")
print(f"• Más ligero: {mas_ligero['modelo']} ({mas_ligero['parametros_M']:.1f} M de parámetros, {mas_ligero['tamano_MB_fp32']:.0f} MB).")
if mejor_f1["modelo"] != mas_rapido["modelo"]:
    print(f"• El más preciso ({mejor_f1['modelo']}) no es el más rápido ({mas_rapido['modelo']}).")
else:
    print(f"• {mejor_f1['modelo']} es a la vez el más preciso y el más rápido.")

## 12. Guardar entregables

In [ ]:
from importlib.metadata import version

paquetes = ["torch", "transformers", "datasets", "huggingface_hub", "scikit-learn", "pandas", "numpy", "matplotlib", "openpyxl", "pillow"]
lineas = [f"{p}=={version(p)}" for p in paquetes]
(RESULTADOS / "requirements_colab.txt").write_text("\n".join(lineas) + "\n")
(RESULTADOS / "entorno.json").write_text(json.dumps({
    "python": sys.version.split()[0],
    "gpu": torch.cuda.get_device_name(0) if HAY_GPU else None,
    "semilla": SEMILLA, "modelos": MODELOS,
}, indent=2, ensure_ascii=False))
predicciones = dfs["test"][["id", "clase", "etiqueta"]].copy()
for nombre, p in probs.items():
    predicciones[f"prob_bache_{nombre}"] = p
predicciones.to_csv(RESULTADOS / "predicciones_test.csv", index=False)

destino = Path("/content/entregables_urbanos.zip")
with zipfile.ZipFile(destino, "w", zipfile.ZIP_DEFLATED) as z:
    for ruta in PROYECTO.rglob("*"):
        if ruta.is_file() and RAW not in ruta.parents and "__pycache__" not in ruta.parts:
            z.write(ruta, ruta.relative_to(PROYECTO))
print("ZIP listo:", destino, f"({destino.stat().st_size / 1e6:.1f} MB)")
print("\n".join(lineas))

try:
    from google.colab import files
    files.download(str(destino))
except Exception:
    pass